## Assignment #1 

In [65]:
import pandas as pd 
import yaml as yl
from pathlib import Path
import re

In [66]:
df_2023 = pd.read_csv(r"../Dataminers2023.csv")
df_2024 = pd.read_csv(r"../Dataminers2024.csv")
df_2025 = pd.read_csv(r"../Dataminers2025.csv")
df_2026 = pd.read_csv(r"../Dataminers2026.csv")

In [67]:
#BASE = Path(r"Assignment 01/Assignment #01 (Dataset)/Cleaned") 
canonical = ["timestamp", "password", "shoe_size", "programme",
             "height", "why", "seattle"]

# ----- Merging the files -----
renames = {
    2023: {
        "Timestamp": "timestamp",
        "Your mean shoe size (In European Continental system)  ": "shoe_size",
        "Which programme are you studying?": "programme",
        "Your height (in International inches)": "height",
        "Why are you taking this course?": "why",
        'How many letters are there in the word "Seattle"?': "seattle",
    },
    2025: {
        "Timestamp": "timestamp",
        "Enter_Your_Password": "password",
        "Your_mean_shoe_size__In_the_European_Continental_system_": "shoe_size",
        "Which_program_are_you_studying_": "programme",
        "Your_height__in_International_inches_": "height",
        "Why_are_you_taking_this_course_": "why",
        "How_many_letters_are_there_in_the_word__Seattle__": "seattle",
    },
}
# 2024 and 2026 share identical headers, so one mapping covers both
renames[2024] = renames[2026] = {
    "Timestamp": "timestamp",
    "Enter Your Password": "password",
    "Your mean shoe size (In the European Continental system)": "shoe_size",
    "Which program are you studying?": "programme",
    "Your height (in International inches)": "height",
    "Why are you taking this course?": "why",
    'How many letters are there in the word "Seattle"?': "seattle",
}

 
frames = []
for year, df in [(2023, df_2023), (2024, df_2024), (2025, df_2025), (2026, df_2026)]:
    df = df.rename(columns=renames[year])
 
    df["source_file"] = year      # which file the row came from, NOT the response year
    frames.append(df)

    

combined = pd.concat(frames, ignore_index=True)
combined["response_year"] = combined["timestamp"].str.extract(r"(\d{4})").astype(int)
combined = combined[canonical + ["source_file", "response_year"]] 

combined = combined.drop(columns="password")
 


# ----- Height -> cm -----
FEET_RE = re.compile(r"^(\d+)\s*['\u2019]+\s*(\d+)?")
 
def to_cm(v):
    """Return height in cm, or None if the value carries no usable height."""
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
 
    # a) feet'inches notation: 5'10", 6''2, 5'10
    m = FEET_RE.match(s)
    if m:
        feet = int(m.group(1))
        inches = int(m.group(2) or 0)
        if feet > 8:          # 70'1 -- stray apostrophe after an inches value
            return None       # DECISION: refuse rather than guess
        return (feet * 12 + inches) * 2.54
 
    # b) strip unit words, normalise the Danish decimal comma
    explicit_cm = "cm" in s
    s = re.sub(r"(cm|centimeters?|centimetres?|inches|inch|in|m)\b", "", s).strip()
    s = s.replace(",", ".")
 
    # c) anything still non-numeric carries no height
    try:
        x = float(s)
    except ValueError:
        return None
 
    # d) classify by magnitude
    if explicit_cm or x > 100:
        return x                      # already cm
    if x < 2:
        return x * 100                # metres
    if x < 10:
        return x * 30.48              # feet, decimal (DECISION: 5,9 -> 5.9 ft)
    return x * 2.54                   # inches, as the question asked
 
combined["height_cm"] = combined["height"].map(to_cm).round(1)


# ----- Shoe size -----
def shoe_eu(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    s = re.sub(r"(eu|euro|european|us|uk|size)", "", s).strip()   # strip unit words
    s = s.replace(",", ".")
    s = re.sub(r"^[^\d]+", "", s)      # drop anything before the first digit
    try:
        x = float(s)
    except ValueError:
        return None
    if x < 20:                      # a US size, if one ever appears
        return round(x + 33, 1)
    if x > 60:                      # nonsense upper bound
        return None
    return x

combined["shoe_eu"] = combined["shoe_size"].map(shoe_eu)
combined["shoe_eu"] = pd.to_numeric(combined["shoe_eu"])

 
# ----- Programme -----
def tidy_programme(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower().replace("digitial", "digital")
    if "software design" in s:      return "Software Design"
    if "digital innovation" in s:   return "Digital Innovation and Management"
    if "digital design" in s:       return "Digital Design"
    if "games" in s:                return "Games"
    if "computer science" in s:     return "Computer Science"
    if "elsewhere" in s:            return "Elsewhere (non-ITU)"
    return "Other"
 
combined["programme_clean"] = combined["programme"].map(tidy_programme)

# Dropping the useless row
junk = combined["height"].astype(str).str.strip().str.lower() == "1 ton"
"dropping", junk.sum(), "row(s):"
combined.loc[junk, ["height", "shoe_size", "seattle", "why"]].to_string()

combined = combined[~junk].reset_index(drop=True)
assert len(combined) == 222, len(combined)

 
# ----- Impute height -----
combined["height_imputed"] = combined["height_cm"].isna()
combined["height_cm"] = combined["height_cm"].fillna(
    round(combined["height_cm"].median(), 1)
)


# ----- WHY COLUMN -----
# Normalize text while preserving the original column
combined["why_normalized"] = (
    combined["why"]
    .fillna("")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Define the motivation options
motivation_options = {
    "interest": [
        "it is the most exciting",
        "it seemed interesting",
        "Machinelearning is awesome"
        
    ],
    "mandatory": [
        "this was mandatory for me",
        "The other courses were even less attractive",
        "Specialisation",
        "It is a part of my specialization, and I felt good about picking a ready-made specialization direction"
    ],

    "commercial_skills": [
        "it'll give me commercially useful skills",
        "it'll give me business skills",
        "it'll give me thesis relevant skills",
        "I am interested in natural science & engineering and data mining is a good tool for these fields",
        "I wish to become the Data Mining Guru at my current workplace",
        "to solve/optimise issues in the construction industry",
    ],
    "uncertain": [
        "i'm not sure, let's see how it works out",
    ], 
    "garbage": [
        "My favorite ice cream is cheese pie",
        "Data Attraction",
        "Leon asked me to help out"
    ]
}


# Create one binary feature per motivation
for feature, phrases in motivation_options.items():
    combined[feature] = (
        combined["why_normalized"]
        .apply(
            lambda answer: int(
                any(phrase.lower() in answer for phrase in phrases)
            )
        )
    )

# ----- TIMESTAMP -----
ts = (combined["timestamp"].astype("string").str.strip()
        .str.replace(r"\s+(GMT[+-]\d{1,2}|[A-Z]{2,5})$", "", regex=True))

DAYFIRST = {2023: True, 2024: False, 2025: False, 2026: False}

parsed = pd.Series(index=combined.index, dtype="object")
for src, dayfirst in DAYFIRST.items():
    m = combined["source_file"] == src
    parsed[m] = ts[m].apply(
        lambda x: pd.to_datetime(x, format="mixed", dayfirst=dayfirst, errors="coerce"))

parsed = parsed.apply(lambda x: x.replace(tzinfo=None) if pd.notna(x) and x.tzinfo else x)

combined["timestamp"] = pd.to_datetime(parsed, errors="coerce")
combined["timestamp_clean"] = combined["timestamp"].dt.strftime("%d/%m/%Y %H:%M")

# Convert to Copenhagen local time.
#   2023 no zone recorded -> assumed already local
#   2024 GMT+3  -> -1h     2025 +09:00 -> -7h     2026 PDT (UTC-7) -> +9h
SHIFT = {2023: 0, 2024: -1, 2025: -7, 2026: 9}

combined["timestamp_cph"] = combined["timestamp"] + combined["source_file"].map(
    lambda y: pd.Timedelta(hours=SHIFT[y])
)

combined["hour"] = combined["timestamp_cph"].dt.hour        # <-- add this

combined["time_of_day"] = pd.cut(
    combined["hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=["Night", "Morning", "Afternoon", "Evening"],
)

# Days between a respondent's submission and the first response in their cohort.
# Grouped by response_year
combined["days_after_open"] = (
    combined["timestamp_cph"]
    - combined.groupby("response_year")["timestamp_cph"].transform("min")
).dt.days

# Create a binary variable for the free-choice cohort
combined["free_choice"] = (combined["response_year"] == 2026).astype(int)

# ----- DMI API TIMESTAMPS -----
CPH = "Europe/Copenhagen"
LOC = dict(ambiguous="NaT", nonexistent="shift_forward")

ts_utc = combined["timestamp_cph"].dt.tz_localize(CPH, **LOC).dt.tz_convert("UTC")
combined["timestamp_api"] = ts_utc.dt.strftime("%Y-%m-%dT%H:%M:%SZ")

# Optional: day window for daily values
def to_utc_iso(s):
    return (s.dt.tz_localize(CPH, **LOC)
             .dt.tz_convert("UTC")
             .dt.strftime("%Y-%m-%dT%H:%M:%SZ"))

local_midnight = combined["timestamp_cph"].dt.normalize()
combined["dmi_datetime"] = (to_utc_iso(local_midnight) + "/" +
                            to_utc_iso(local_midnight + pd.DateOffset(days=1)))

### Sammensætte en CSV fil med de kolonner man vil have

In [68]:
final_cols = [
    "source_file", "response_year",
    "timestamp_cph", "timestamp_api", "dmi_datetime",
    "time_of_day", "days_after_open",
    "height_cm",
    "shoe_eu",
    "programme_clean", "free_choice",
    "seattle",
    "why", "interest", "mandatory", "commercial_skills", "uncertain"
]
clean = combined[final_cols]
clean.to_csv("dataminers_cleaner.csv", index=False)

### Helpers to feature engineer Why column

In [69]:
combined["why"].value_counts(dropna=False)

why
It is the most exciting                                                                                                                                                                45
This was mandatory for me                                                                                                                                                              26
It is the most exciting;It'll give me commercially useful skills                                                                                                                       23
It'll give me commercially useful skills                                                                                                                                               23
I'm not sure, let's see how it works out                                                                                                                                               13
It'll give me business skills                                     

In [70]:
df_2023["Why are you taking this course?"].unique()

array(["It'll give me business skills",
       "It is the most exciting, This was mandatory for me, It'll give me business skills",
       'This was mandatory for me',
       "It is the most exciting, It'll give me business skills",
       "This was mandatory for me, It'll give me business skills",
       "The other courses were even less attractive, It is the most exciting, It'll give me business skills",
       'It is the most exciting',
       'It is the most exciting, This was mandatory for me',
       'Leon asked me to help out',
       "I'm not sure, let's see how it works out, It'll give me business skills",
       'I am interested in natural science & engineering and data mining is a good tool for these fields',
       "The other courses were even less attractive, I'm not sure, let's see how it works out",
       "The other courses were even less attractive, It is the most exciting, This was mandatory for me, I'm not sure, let's see how it works out, It'll give me business skil

In [71]:
df_2024["Why are you taking this course?"].unique()

array(['It is the most exciting',
       "It'll give me commercially useful skills",
       "I'm not sure, let's see how it works out",
       "This was mandatory for me;It'll give me commercially useful skills",
       "It is the most exciting;It'll give me commercially useful skills",
       'This was mandatory for me;Would it be disruptive to have an SQL injection here?',
       "It is the most exciting;I'm not sure, let's see how it works out",
       'This was mandatory for me',
       'It is the most exciting;This was mandatory for me',
       "It'll give me commercially useful skills;It's my specialisation",
       'to solve/optimise issues in the construction industry',
       "I'm not sure, let's see how it works out;It'll give me commercially useful skills",
       'The other courses were even less attractive',
       "The other courses were even less attractive;It is the most exciting;I'm not sure, let's see how it works out;It'll give me commercially useful skills",
       

In [72]:
df_2025["Why_are_you_taking_this_course_"].unique()

array(["It is the most exciting;This was mandatory for me;It'll give me commercially useful skills",
       'It is the most exciting',
       "It is the most exciting;It'll give me commercially useful skills",
       'It is the most exciting;This was mandatory for me;I love the idea of data and the word "mining" combined ',
       'This was mandatory for me',
       "I'm not sure, let's see how it works out;It'll give me commercially useful skills",
       "It'll give me commercially useful skills",
       "I'm not sure, let's see how it works out",
       "I'm not sure, let's see how it works out;data skills",
       "It is the most exciting;I'm not sure, let's see how it works out",
       'It is the most exciting;It is part of my specialization',
       'It is the most exciting;This was mandatory for me',
       'My favorite ice cream is cheese pie',
       "It'll give me commercially useful skills;Part of potential specialization",
       "The other courses were even less attractiv

### Feature engineer weather data: daily mm of rain

We store the daily rain in a seperate csv, this is to limit the amount of api calls, so instead of calling the api for each row, we just to once per date. The two tables can be joined on the date dolumn. 

Weather station is currently set to Copenhagen airport. Full overview of weather stations, can be found [here](https://www.dmi.dk/friedata/dokumentation/data/meteorological-observation-data-stations)

This has to be run after the dataminers_cleaner.csv has been created

In [ ]:
import time
import requests

BASE_URL = "https://opendataapi.dmi.dk/v2/climateData/collections/stationValue/items"
STATION_ID = "06180"        # Copenhagen Airport (Kastrup)
PARAMETER = "acc_precip"    # accumulated precipitation, mm
CPH = "Europe/Copenhagen"
FMT = "%Y-%m-%dT%H:%M:%SZ"  # time format, date

def get_daily_rain_mm(date):
    """Return mm of rain for one Copenhagen calendar day (a date or 'YYYY-MM-DD')."""
    day_start = pd.Timestamp(date).tz_localize(CPH).tz_convert("UTC")
    day_end = (pd.Timestamp(date) + pd.DateOffset(days=1)).tz_localize(CPH).tz_convert("UTC")

    params = {
        "stationId": STATION_ID,
        "parameterId": PARAMETER,
        "timeResolution": "day",
        "datetime": f"{day_start.strftime(FMT)}/{day_end.strftime(FMT)}",
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()

    for f in r.json().get("features", []):
        p = f["properties"]
        start = pd.to_datetime(p["from"], utc=True)
        if day_start <= start < day_end: # start included, end excluded (DMI is inclusive as default)
            return p["value"]
    return None   # Returns none if DMI has no value for that day


# Load the dates from the dataminers_cleaner.csv
df = pd.read_csv("dataminers_cleaner.csv", parse_dates=["timestamp_cph"])
dates = sorted(df["timestamp_cph"].dropna().dt.date.unique())


# Fetch rainfall per date
rows = []
for d in dates:
    rows.append({"date": d, "mm_rain": get_daily_rain_mm(d)})
    time.sleep(0.1)   # this is good practise when working with APIs

date_rainfall = pd.DataFrame(rows)
date_rainfall.to_csv("date_rainfall.csv", index=False)


45 unique dates
         date  mm_rain
0  2020-08-26      3.7
1  2020-08-27     11.3
2  2020-09-01      0.5
3  2020-09-02      0.3
4  2021-08-30      0.0
Missing: 0
